# FilPHANGS - Synthetic Map Figure Production

Generates figures comparing the PSF synthetic filament maps to the original JWST images:

1. **Reconstruction comparison** -- original / reconstructed / residual three-panel figure
2. **Flux fraction analysis** -- what percentage of the original image flux each scale accounts for
3. **RGB multi-scale composite** -- each scale mapped to a colour channel to show spatial hierarchy
4. **Multi-galaxy panel** -- assembles individual composites into a publication panel

**Run order**: Cell 1 (utilities) -> Cells 2-5 (figures, independently runnable after Cell 1)


In [ ]:
# =============================================================================
# Cell 1: Imports and shared utilities
# Run this cell first -- defines load_fits, normalize, apply_color used everywhere.
# =============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from astropy.io import fits
from pathlib import Path
from scipy.ndimage import zoom

# -- Paths: edit once here ----------------------------------------------------
BASE_DIR   = Path(r"C:\Users\jhoffm72\Documents\FilPHANGS\Data")
OUTPUT_DIR = BASE_DIR / "Figures"
OUTPUT_DIR.mkdir(exist_ok=True)

# Folders to exclude when scanning for galaxy data
EXCLUDED = {"Figures","OriginalImages","masks_v5_simple","IC5146_PSW",
            "ngc2090_F555W","ngc0628_F2100W"}

# -- Colour assignments for multi-scale RGB composites ------------------------
# Scale tag in filename -> (RGB list, display colour name, display label)
# Blue = large-scale | Red = mid-scale | Green = fine-scale filaments
SCALE_COLORS = {
    "0128pc": ([0.0, 0.0, 1.0], "blue",  "128 pc"),
    "0064pc": ([1.0, 0.0, 0.0], "red",   "64 pc"),
    "0032pc": ([0.0, 1.0, 0.0], "green", "32 pc"),
}
CDD_SCALE_BINS = [16, 32, 64, 128, 256]

# =============================================================================
# Image utilities (used by all subsequent cells)
# =============================================================================

def load_fits(path):
    """Load a FITS primary HDU, replacing NaNs with zero."""
    with fits.open(path, ignore_missing=True) as h:
        return np.nan_to_num(np.array(h[0].data, dtype=float))

def normalize(data, percentile=99, stretch=20):
    """
    Stretch data to [0, stretch], clipping at the given percentile.
    percentile: upper brightness clip (e.g. 99 ignores the top 1% of pixels)
    stretch: linear scale factor applied after normalisation
    """
    dmin = np.min(data)
    dmax = np.percentile(data, percentile)
    return np.clip((data - dmin) / (dmax - dmin + 1e-9), 0, 1) * stretch

def resample_to(data, target_shape):
    """Bilinear zoom to target_shape. No-op if shapes already match."""
    if data.shape == target_shape: return data
    return zoom(data,
                (target_shape[0] / data.shape[0], target_shape[1] / data.shape[1]),
                order=1)

def apply_color(data, color_rgb, brightness=2.0, gamma=0.4):
    """
    Tint a single-channel data array with an RGB colour.
    gamma < 1 lifts faint features; brightness scales the overall level.
    Returns an (H, W, 3) float32 RGB array.
    """
    scaled = np.clip((np.clip(data, 0, None) ** gamma) * brightness, 0, 1)
    rgb    = np.zeros(data.shape + (3,), dtype=np.float32)
    for c in range(3):
        rgb[..., c] = scaled * color_rgb[c]
    return rgb

def get_galaxy_folders():
    """Return all galaxy folder names, skipping non-galaxy directories."""
    return [f for f in os.listdir(BASE_DIR)
            if (BASE_DIR / f).is_dir()
            and f not in EXCLUDED
            and not f.endswith((".txt", ".xlsx"))]


In [ ]:
# =============================================================================
# Cell 2: Build Reconstruction + Three-Panel Comparison Figure
# Sums all per-scale synthetic maps into a reconstructed image, saves it to
# FITS, then displays original / reconstructed / absolute residual side by side.
# Edit Region and Band to switch galaxies.
# =============================================================================
Region    = "ngc0628"
Band      = "F770W"
synth_dir = BASE_DIR / f"{Region}_{Band}" / "SyntheticMap"
orig_path = BASE_DIR / "OriginalImages" / f"{Region}_{Band}_JWST_Emission_starsub.fits"

orig  = load_fits(orig_path)
recon = np.zeros_like(orig)

for fname in os.listdir(synth_dir):
    if not fname.endswith(".fits"): continue
    if any(x in fname for x in ("Reconstruction", "Residual")): continue
    try:
        img = load_fits(synth_dir / fname)
        if img.shape != orig.shape:
            print(f"  Shape mismatch, skipping: {fname}"); continue
        recon += img
    except Exception as e:
        print(f"  {fname}: {e}")

# Save reconstruction and residual FITS files
hdr = fits.getheader(orig_path)
fits.PrimaryHDU(recon, hdr).writeto(
    synth_dir / f"Reconstruction_{Region}_{Band}.fits", overwrite=True)
fits.PrimaryHDU(np.abs(orig - recon), hdr).writeto(
    synth_dir / f"Residual_{Region}_{Band}.fits", overwrite=True)
print(f"Reconstruction total flux: {np.sum(recon):.3e}")
print(f"Residual total flux:       {np.sum(np.abs(orig-recon)):.3e}")

# -- Three-panel comparison figure --------------------------------------------
def crop_center(img, frac=0.5):
    h, w = img.shape
    s = int(min(h, w) * frac)
    return img[h//2-s//2:h//2+s//2, w//2-s//2:w//2+s//2]

panels = [crop_center(orig), crop_center(recon), crop_center(np.abs(orig-recon))]
titles = ["Original Image", "Reconstructed Image", "Absolute Residual"]
flat   = np.concatenate([p.flatten() for p in panels])
vmin, vmax = np.percentile(flat[flat > 0], [1, 99]) if np.any(flat > 0) else (0, 1)

fig, axs = plt.subplots(1, 3, figsize=(18, 6), facecolor="white",
                        gridspec_kw={"wspace": 0.04})
for ax, img, title in zip(axs, panels, titles):
    ax.imshow(img, cmap="viridis", origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=12, pad=6)
    ax.axis("off")

fig.suptitle(f"{Region} {Band} -- Synthetic Map Reconstruction", fontsize=14, y=1.01)
out = OUTPUT_DIR / f"Reconstruction_{Region}_{Band}.png"
for ext in (".png", ".pdf"):
    fig.savefig(str(out).replace(".png", ext), dpi=300,
                bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved {out}")


In [ ]:
# =============================================================================
# Cell 3: Flux Fraction Analysis
# For each galaxy, computes what percentage of the original image's total flux
# each CDD scale of synthetic filament maps accounts for.
# Saves per-galaxy bar charts and one combined chart for the full sample.
# =============================================================================

def flux_bar_chart(scale_flux_pct, ax, title="", skip_empty=True):
    """
    Bar chart of flux fraction (%) per CDD scale, with a cumulative line.

    scale_flux_pct : dict mapping scale_int -> percent_of_original_flux
    skip_empty     : if True, omit scales with 0% flux from x-axis
    """
    bins = [b for b in CDD_SCALE_BINS
            if not skip_empty or scale_flux_pct.get(b, 0) > 0]
    pcts = np.array([scale_flux_pct.get(b, 0) for b in bins])
    pos  = np.arange(len(bins))

    for i in range(len(bins)):      # gray 100% reference bars
        ax.bar(pos[i], 100, color="lightgray", width=0.6, edgecolor="black", zorder=1)
    for i, p in enumerate(pcts):   # coloured flux bars
        ax.bar(pos[i], p, color="steelblue", alpha=0.75, width=0.6,
               edgecolor="black", zorder=2)
    ax.plot(pos, np.cumsum(pcts), marker="o", color="black",
            lw=2, zorder=3, label="Cumulative")

    ax.set_xticks(pos)
    ax.set_xticklabels([str(b) for b in bins])
    ax.set_xlabel("Scale (pc)")
    ax.set_ylabel("% of original image flux")
    ax.set_ylim(0, 110)
    ax.set_title(title, fontsize=10)
    ax.grid(True, axis="y", ls="--", alpha=0.6)
    ax.legend(fontsize=8)

galaxy_flux_pcts = {}
combined_synth   = {b: 0.0 for b in CDD_SCALE_BINS}
combined_orig    = 0.0

for folder in get_galaxy_folders():
    synth_dir = BASE_DIR / folder / "SyntheticMap"
    if not synth_dir.is_dir(): continue

    # Find original image for this galaxy
    orig_path_g = next(
        (BASE_DIR / "OriginalImages" / f
         for f in os.listdir(BASE_DIR / "OriginalImages")
         if folder.split("_")[0].lower() in f.lower() and f.endswith(".fits")),
        None)
    if orig_path_g is None: continue

    try:
        orig_flux = float(np.nansum(load_fits(orig_path_g)))
    except Exception as e:
        print(f"  {folder} orig: {e}"); continue
    if orig_flux <= 0: continue

    scale_flux = {b: 0.0 for b in CDD_SCALE_BINS}
    for fname in os.listdir(synth_dir):
        if not fname.endswith(".fits"): continue
        for b in CDD_SCALE_BINS:
            # Match by scale tag (e.g. "0032pc" or "32pc" or "_32pc")
            if f"{b:04d}pc" in fname or f"0{b:03d}pc" in fname or f"_{b}pc" in fname:
                try:
                    flux = float(np.nansum(load_fits(synth_dir / fname)))
                    scale_flux[b] += flux
                    combined_synth[b] += flux
                except Exception as e:
                    print(f"  {fname}: {e}")
                break

    combined_orig += orig_flux
    galaxy_flux_pcts[folder] = {b: 100*v/orig_flux for b,v in scale_flux.items()}

# Per-galaxy bar charts (saved but not displayed to avoid flooding output)
for gal, pcts in sorted(galaxy_flux_pcts.items()):
    fig, ax = plt.subplots(figsize=(7, 5))
    flux_bar_chart(pcts, ax, title=gal, skip_empty=True)
    out = OUTPUT_DIR / f"FluxFraction_{gal}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  {out}")

# Combined chart for the full sample
combined_pcts = {b: 100*combined_synth[b]/combined_orig for b in CDD_SCALE_BINS}
fig, ax = plt.subplots(figsize=(7, 5))
flux_bar_chart(combined_pcts, ax,
               title=f"All galaxies combined (N={len(galaxy_flux_pcts)})",
               skip_empty=False)
out = OUTPUT_DIR / "FluxFraction_combined.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {out}")


In [ ]:
# =============================================================================
# Cell 4: Multi-Scale RGB Composite -- All Galaxies
# Assigns each CDD scale to a colour channel and blends with max compositing.
#
# Colour key: Blue = 128 pc | Red = 64 pc | Green = 32 pc
# Rationale: large filaments appear blue, fine-scale structure appears red/green.
# =============================================================================
PERCENTILE = 99   # upper clip percentile for brightness normalisation
STRETCH    = 20   # linear stretch factor applied after normalisation

legend_handles = [
    mpatches.Patch(color=meta[1], label=meta[2])
    for meta in SCALE_COLORS.values()
]

for folder in get_galaxy_folders():
    synth_dir = BASE_DIR / folder / "SyntheticMap"
    if not synth_dir.is_dir():
        print(f"  {folder}: no SyntheticMap, skipping"); continue

    # Find one FITS file per scale key
    files = {key: None for key in SCALE_COLORS}
    for fname in os.listdir(synth_dir):
        if not fname.endswith(".fits"): continue
        for key in SCALE_COLORS:
            if key in fname and files[key] is None:
                files[key] = synth_dir / fname

    missing = [k for k, v in files.items() if v is None]
    if missing:
        print(f"  {folder}: missing scales {missing}, skipping"); continue

    # Load and normalise each scale channel
    loaded = {k: normalize(load_fits(v), PERCENTILE, STRETCH)
              for k, v in files.items()}

    # Resample all channels to the largest shape so they align spatially
    # (different scales may have slightly different pixel dimensions)
    target = max(loaded.values(), key=lambda a: a.size).shape
    loaded = {k: resample_to(a, target) for k, a in loaded.items()}

    # Colourize and blend with per-pixel maximum across channels
    channels  = [apply_color(loaded[k], meta[0]) for k, meta in SCALE_COLORS.items()]
    composite = np.clip(np.max(np.stack(channels), axis=0), 0, 1)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(composite, origin="lower")
    ax.axis("off")
    ax.set_title(folder, fontsize=11)
    ax.legend(handles=legend_handles, loc="lower right", framealpha=0.6, fontsize=9)

    out = OUTPUT_DIR / f"{folder}_MultiScaleComposite.png"
    fig.savefig(out, dpi=300, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    print(f"  {folder}")

print("\nDone. All composites saved to:", OUTPUT_DIR)


In [ ]:
# =============================================================================
# Cell 5: Multi-Galaxy Publication Panel
# Assembles composite PNGs saved by Cell 4 into a single publication-ready figure.
# Edit panel_galaxies to choose which galaxies appear and in what order.
# =============================================================================
import matplotlib.image as mpimg

# Edit this list to include any galaxies processed by Cell 4
panel_galaxies = [
    "ngc0628_F770W", "ngc1433_F770W", "ngc2835_F770W", "ngc4254_F770W",
]
titles = [g.replace("_F770W", "").upper() for g in panel_galaxies]

fig, axes = plt.subplots(1, len(panel_galaxies),
                         figsize=(5 * len(panel_galaxies), 5))
if len(panel_galaxies) == 1:
    axes = [axes]

for ax, gal, title in zip(axes, panel_galaxies, titles):
    img_path = OUTPUT_DIR / f"{gal}_MultiScaleComposite.png"
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(title, fontsize=14)
    else:
        ax.text(0.5, 0.5, f"Run Cell 4 first\n({gal})",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=9, color="red")
    ax.axis("off")

plt.tight_layout()
out = OUTPUT_DIR / "MultiScale_Panel.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")
